# 08 - Train and evaluate a Darija GPT

This notebook assembles a small decoder-only language model from classes, trains it on the prepared Darija corpus, and measures its performance on held-out text. It reports validation cross-entropy and perplexity, saves the best checkpoint, and generates a short continuation. The model starts with random weights; no pretrained model is used.

Run notebooks 01 and 02 first to create the processed corpus and tokenizer. This notebook uses the existing training and validation split, and does not download data.

## 1. Load the tokenizer and prepared documents

The tokenizer is fit only on the training partition in notebook 02. Here we load it together with the two text splits and reserve [EOS] to mark document endings.

In [1]:
import copy
import hashlib
import json
import math
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tokenizers import Tokenizer

SEED = 123
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook from the darija-llm-lab project directory.")

PROJECT_ROOT = find_project_root()
TOKENIZER_PATH = PROJECT_ROOT / "artifacts/tokenizer/tokenizer_darija.json"
TRAIN_PATH = PROJECT_ROOT / "data/processed/train.jsonl"
VALIDATION_PATH = PROJECT_ROOT / "data/processed/validation.jsonl"
CHECKPOINT_DIR = PROJECT_ROOT / "artifacts/checkpoints"

required_paths = [TOKENIZER_PATH, TRAIN_PATH, VALIDATION_PATH]
missing = [str(path.relative_to(PROJECT_ROOT)) for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Required files are missing: " + ", ".join(missing) + ". Run notebooks 01 and 02 first."
    )

def read_jsonl_documents(path):
    documents = []
    with path.open(encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, start=1):
            if not line.strip():
                continue
            row = json.loads(line)
            text = row.get("text") if isinstance(row, dict) else None
            if not isinstance(text, str) or not text.strip():
                raise ValueError(f"{path.name}, line {line_number}: expected a non-empty text field.")
            documents.append(text)
    if not documents:
        raise ValueError(f"{path.name} contains no documents.")
    return documents

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
tokenizer.encode_special_tokens = True
eos_token_id = tokenizer.token_to_id("[EOS]")
if eos_token_id is None:
    raise ValueError("The tokenizer must contain the [EOS] special token.")

train_documents = read_jsonl_documents(TRAIN_PATH)
validation_documents = read_jsonl_documents(VALIDATION_PATH)
print("Vocabulary size:", tokenizer.get_vocab_size())
print("Training documents:", len(train_documents))
print("Validation documents:", len(validation_documents))
print("Example:", train_documents[0][:180])

C:\Users\sail_\Documents\Codex\2026-10-03\https-github-com-rasbt-llms-from\outputs\darija-llm-lab\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:377: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:82.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


Vocabulary size: 2000
Training documents: 900
Validation documents: 100
Example: ...عزيزة حلاة فما اكتر ونطقات ايواااتنهدات وقالت ماجاوبش واخا عييت نلح عليه حدو گاليا باللي انا كايبغييني وهادا المهمردت اييواااانسيمة وصافي ههه تخشيت فيه وبدا يخبر علياا حتا نعستج


## 2. Build context windows without crossing documents

Each dataset item contains an input sequence and the same sequence shifted one token to the right as its target. Documents are tokenized independently and end with [EOS], so windows never join the end of one document to the start of another.

In [2]:
class DarijaGPTDataset(Dataset):
    def __init__(self, documents, tokenizer, context_length, stride, eos_token_id):
        if context_length < 2 or stride < 1:
            raise ValueError("context_length must be at least 2 and stride must be positive.")
        self.examples = []
        self.context_length = context_length

        for document in documents:
            token_ids = tokenizer.encode(document).ids + [eos_token_id]
            for start in range(0, len(token_ids) - context_length, stride):
                window = token_ids[start:start + context_length + 1]
                if len(window) == context_length + 1:
                    self.examples.append(torch.tensor(window, dtype=torch.long))

        if not self.examples:
            raise ValueError(
                "No complete context windows were created. Reduce context_length or prepare longer documents."
            )

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        window = self.examples[index]
        return window[:-1], window[1:]

CONTEXT_LENGTH = 64
BATCH_SIZE = 8
WINDOW_STRIDE = 32

train_dataset = DarijaGPTDataset(
    train_documents, tokenizer, CONTEXT_LENGTH, WINDOW_STRIDE, eos_token_id
)
validation_dataset = DarijaGPTDataset(
    validation_documents, tokenizer, CONTEXT_LENGTH, WINDOW_STRIDE, eos_token_id
)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False, num_workers=0
)
validation_loader = DataLoader(
    validation_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=0
)

sample_x, sample_y = next(iter(train_loader))
assert sample_x.shape == sample_y.shape
assert torch.equal(sample_x[:, 1:], sample_y[:, :-1])
print("Training windows:", len(train_dataset))
print("Validation windows:", len(validation_dataset))
print("Batch shapes:", tuple(sample_x.shape), tuple(sample_y.shape))

Training windows: 12243
Validation windows: 1382
Batch shapes: (8, 64) (8, 64)


## 3. Define the GPT configuration and model classes

The configuration keeps model dimensions together. Token embeddings map vocabulary IDs to vectors; learned position embeddings mark sequence order. Transformer blocks process those vectors with causal attention and a feed-forward network. The output head returns one vocabulary score for every position.

In [3]:
@dataclass(frozen=True)
class GPTConfig:
    vocab_size: int
    context_length: int = 64
    embedding_dim: int = 64
    num_layers: int = 2
    num_heads: int = 4
    dropout: float = 0.1
    qkv_bias: bool = False

    def __post_init__(self):
        if self.embedding_dim % self.num_heads != 0:
            raise ValueError("embedding_dim must be divisible by num_heads.")
        if min(self.vocab_size, self.context_length, self.embedding_dim, self.num_layers, self.num_heads) < 1:
            raise ValueError("Model dimensions must be positive.")

model_config = GPTConfig(
    vocab_size=tokenizer.get_vocab_size(),
    context_length=CONTEXT_LENGTH,
    embedding_dim=64,
    num_layers=2,
    num_heads=4,
    dropout=0.1,
)
print(model_config)

GPTConfig(vocab_size=2000, context_length=64, embedding_dim=64, num_layers=2, num_heads=4, dropout=0.1, qkv_bias=False)


In [4]:
class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        if config.embedding_dim % config.num_heads != 0:
            raise ValueError("embedding_dim must be divisible by num_heads.")

        self.num_heads = config.num_heads
        self.head_dim = config.embedding_dim // config.num_heads
        self.embedding_dim = config.embedding_dim
        self.query = nn.Linear(config.embedding_dim, config.embedding_dim, bias=config.qkv_bias)
        self.key = nn.Linear(config.embedding_dim, config.embedding_dim, bias=config.qkv_bias)
        self.value = nn.Linear(config.embedding_dim, config.embedding_dim, bias=config.qkv_bias)
        self.output = nn.Linear(config.embedding_dim, config.embedding_dim)
        self.attention_dropout = nn.Dropout(config.dropout)
        mask = torch.triu(
            torch.ones(config.context_length, config.context_length, dtype=torch.bool), diagonal=1
        )
        self.register_buffer("future_mask", mask)

    def forward(self, x):
        batch_size, sequence_length, _ = x.shape
        if sequence_length > self.future_mask.shape[0]:
            raise ValueError("Input sequence exceeds the configured context length.")

        shape = (batch_size, sequence_length, self.num_heads, self.head_dim)
        query = self.query(x).view(shape).transpose(1, 2)
        key = self.key(x).view(shape).transpose(1, 2)
        value = self.value(x).view(shape).transpose(1, 2)

        scores = (query @ key.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = self.future_mask[:sequence_length, :sequence_length]
        scores = scores.masked_fill(mask, -torch.inf)
        weights = self.attention_dropout(torch.softmax(scores, dim=-1))
        context = weights @ value
        context = context.transpose(1, 2).contiguous().view(
            batch_size, sequence_length, self.embedding_dim
        )
        return self.output(context)


class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(config.embedding_dim, 4 * config.embedding_dim),
            nn.GELU(),
            nn.Linear(4 * config.embedding_dim, config.embedding_dim),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.norm1 = nn.LayerNorm(config.embedding_dim)
        self.attention = MultiHeadCausalSelfAttention(config)
        self.norm2 = nn.LayerNorm(config.embedding_dim)
        self.feed_forward = FeedForward(config)
        self.residual_dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = x + self.residual_dropout(self.attention(self.norm1(x)))
        x = x + self.residual_dropout(self.feed_forward(self.norm2(x)))
        return x


class GPTModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.embedding_dim)
        self.position_embedding = nn.Embedding(config.context_length, config.embedding_dim)
        self.embedding_dropout = nn.Dropout(config.dropout)
        self.blocks = nn.Sequential(*(TransformerBlock(config) for _ in range(config.num_layers)))
        self.final_norm = nn.LayerNorm(config.embedding_dim)
        self.output_head = nn.Linear(config.embedding_dim, config.vocab_size, bias=False)
        self.apply(self._initialize_weights)

    @staticmethod
    def _initialize_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, token_ids):
        if token_ids.ndim != 2:
            raise ValueError("token_ids must have shape (batch, sequence).")
        batch_size, sequence_length = token_ids.shape
        if sequence_length > self.config.context_length:
            raise ValueError("Input sequence exceeds the configured context length.")

        positions = torch.arange(sequence_length, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)[None, :, :]
        x = self.embedding_dropout(x)
        x = self.blocks(x)
        x = self.final_norm(x)
        return self.output_head(x)

## 4. Check the model's forward pass

For every input token the model predicts a score for each vocabulary item. The output therefore has shape (batch, sequence, vocabulary). The parameter count shows the size of this small starting model.

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GPTModel(model_config).to(device)
batch_x, batch_y = next(iter(train_loader))

with torch.no_grad():
    sample_logits = model(batch_x.to(device))

assert sample_logits.shape == (
    batch_x.shape[0], CONTEXT_LENGTH, tokenizer.get_vocab_size()
)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print("Device:", device)
print("Logits:", tuple(sample_logits.shape))
print(f"Trainable parameters: {parameter_count:,}")

Device: cpu
Logits: (8, 64, 2000)
Trainable parameters: 359,808


## 5. Train with next-token cross-entropy

At each position the target is the next token ID. Cross-entropy compares the model's vocabulary scores with those targets. The trainer periodically measures loss on training batches and the full validation split, clips large gradients, and keeps the weights with the best validation loss.

In [6]:
class GPTTrainer:
    def __init__(
        self, model, train_loader, validation_loader, device,
        learning_rate=3e-4, weight_decay=0.1
    ):
        self.model = model
        self.train_loader = train_loader
        self.validation_loader = validation_loader
        self.device = device
        self.optimizer = torch.optim.AdamW(
            model.parameters(), lr=learning_rate, weight_decay=weight_decay
        )

    @torch.no_grad()
    def evaluate(self, data_loader, max_batches=None):
        was_training = self.model.training
        self.model.eval()
        total_loss = 0.0
        total_tokens = 0

        for batch_number, (inputs, targets) in enumerate(data_loader, start=1):
            inputs = inputs.to(self.device)
            targets = targets.to(self.device)
            logits = self.model(inputs)
            loss_sum = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
                reduction="sum",
            )
            total_loss += loss_sum.item()
            total_tokens += targets.numel()
            if max_batches is not None and batch_number >= max_batches:
                break

        self.model.train(was_training)
        if total_tokens == 0:
            raise ValueError("The data loader produced no target tokens.")
        return total_loss / total_tokens

    def fit(self, max_steps=200, eval_interval=25, gradient_clip=1.0, train_eval_batches=10):
        if max_steps < 1 or eval_interval < 1:
            raise ValueError("max_steps and eval_interval must be positive.")

        history = []
        best_validation_loss = float("inf")
        best_state = None
        step = 0

        while step < max_steps:
            saw_batch = False
            for inputs, targets in self.train_loader:
                saw_batch = True
                self.model.train()
                inputs = inputs.to(self.device)
                targets = targets.to(self.device)
                self.optimizer.zero_grad(set_to_none=True)
                logits = self.model(inputs)
                loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten())
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), gradient_clip)
                self.optimizer.step()
                step += 1

                if step == 1 or step % eval_interval == 0 or step == max_steps:
                    train_loss = self.evaluate(
                        self.train_loader, max_batches=train_eval_batches
                    )
                    validation_loss = self.evaluate(self.validation_loader)
                    history.append({
                        "step": step,
                        "train_loss": train_loss,
                        "validation_loss": validation_loss,
                    })
                    print(
                        f"Step {step:>4} | train loss {train_loss:.4f} "
                        f"| validation loss {validation_loss:.4f}"
                    )
                    if validation_loss < best_validation_loss:
                        best_validation_loss = validation_loss
                        best_state = {
                            key: value.detach().cpu().clone()
                            for key, value in self.model.state_dict().items()
                        }

                if step >= max_steps:
                    break

            if not saw_batch:
                raise ValueError("The training data loader produced no batches.")

        if best_state is not None:
            self.model.load_state_dict(best_state)
        return history

## 6. Run a small from-scratch training job

The defaults keep the model modest for a CPU run. Increase MAX_STEPS, model dimensions, and corpus size when more compute is available. Training starts from random initialization; only the training split contributes gradients.

In [7]:
MAX_STEPS = 200
EVAL_INTERVAL = 25
LEARNING_RATE = 3e-4

trainer = GPTTrainer(
    model=model,
    train_loader=train_loader,
    validation_loader=validation_loader,
    device=device,
    learning_rate=LEARNING_RATE,
)
initial_validation_loss = trainer.evaluate(validation_loader)
print(f"Initial validation loss: {initial_validation_loss:.4f}")
history = trainer.fit(max_steps=MAX_STEPS, eval_interval=EVAL_INTERVAL)

Initial validation loss: 7.6132


Step    1 | train loss 7.6078 | validation loss 7.6062


Step   25 | train loss 7.3455 | validation loss 7.3484


Step   50 | train loss 7.1080 | validation loss 7.1143


Step   75 | train loss 6.9550 | validation loss 6.9383


Step  100 | train loss 6.8297 | validation loss 6.8243


Step  125 | train loss 6.7430 | validation loss 6.7601


Step  150 | train loss 6.7568 | validation loss 6.7303


Step  175 | train loss 6.7523 | validation loss 6.7184


Step  200 | train loss 6.7169 | validation loss 6.7135


## 7. Evaluate on held-out Darija text

Validation loss is the mean next-token negative log-likelihood over all target tokens, not an average of batch averages. Perplexity is exp(validation loss) and summarizes the model's uncertainty about the next token on this split. Lower is better for the same tokenizer and evaluation data; perplexity alone does not measure whether generated Darija is useful or fluent.

In [8]:
final_validation_loss = trainer.evaluate(validation_loader)
validation_perplexity = (
    math.exp(final_validation_loss) if final_validation_loss < 20 else float("inf")
)
print(f"Initial validation loss: {initial_validation_loss:.4f}")
print(f"Best-checkpoint validation loss: {final_validation_loss:.4f}")
print(f"Validation perplexity: {validation_perplexity:.2f}")

Initial validation loss: 7.6132
Best-checkpoint validation loss: 6.7135
Validation perplexity: 823.45


## 8. Save the best checkpoint

The checkpoint contains model weights, architecture settings, and the tokenizer checksum. The checksum ties later use of the checkpoint to the vocabulary used during training. Checkpoints are saved under artifacts/ and are not committed to the repository.

In [9]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = CHECKPOINT_DIR / "darija_gpt_best.pt"
checkpoint = {
    "model_state_dict": {
        key: value.detach().cpu() for key, value in model.state_dict().items()
    },
    "model_config": asdict(model_config),
    "training": {
        "max_steps": MAX_STEPS,
        "learning_rate": LEARNING_RATE,
        "best_validation_loss": final_validation_loss,
        "validation_perplexity": validation_perplexity,
        "seed": SEED,
    },
    "tokenizer_sha256": hashlib.sha256(TOKENIZER_PATH.read_bytes()).hexdigest(),
}
torch.save(checkpoint, checkpoint_path)
print("Saved checkpoint:", checkpoint_path.relative_to(PROJECT_ROOT))

Saved checkpoint: artifacts\checkpoints\darija_gpt_best.pt


## 9. Generate a short continuation

Generation repeatedly predicts the next token from the most recent context. Temperature controls sampling randomness; top-k limits sampling to the highest-scoring candidates. The example uses a short validation prompt for inspection only; it is never used for training updates.

In [10]:
@torch.no_grad()
def generate_text(
    model, prompt_ids, max_new_tokens, context_length,
    temperature=0.8, top_k=40, eos_token_id=None
):
    if temperature <= 0:
        raise ValueError("temperature must be positive.")
    model.eval()
    token_ids = prompt_ids

    for _ in range(max_new_tokens):
        context = token_ids[:, -context_length:]
        logits = model(context)[:, -1, :] / temperature
        if top_k is not None:
            current_top_k = min(top_k, logits.size(-1))
            threshold = torch.topk(logits, current_top_k, dim=-1).values[:, -1, None]
            logits = logits.masked_fill(logits < threshold, -torch.inf)

        probabilities = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probabilities, num_samples=1)
        token_ids = torch.cat((token_ids, next_token), dim=1)
        if eos_token_id is not None and next_token.item() == eos_token_id:
            break

    return token_ids

prompt_text = " ".join(validation_documents[0].split()[:8])
prompt_ids = torch.tensor(
    [tokenizer.encode(prompt_text).ids], dtype=torch.long, device=device
)
generated_ids = generate_text(
    model,
    prompt_ids,
    max_new_tokens=40,
    context_length=model_config.context_length,
    temperature=0.8,
    top_k=40,
    eos_token_id=eos_token_id,
)
generated_text = tokenizer.decode(
    generated_ids[0].tolist(), skip_special_tokens=True
)
print("Prompt:", prompt_text)
print("Generated text:", generated_text)

generation_samples = [{
    "prompt": prompt_text,
    "generated_text": generated_text,
    "continuation": generated_text[len(prompt_text):].lstrip()
        if generated_text.startswith(prompt_text) else generated_text,
    "temperature": 0.8,
    "top_k": 40,
}]

for document in validation_documents[1:3]:
    sample_prompt = " ".join(document.split()[:8])
    sample_ids = torch.tensor(
        [tokenizer.encode(sample_prompt).ids], dtype=torch.long, device=device
    )
    sample_output_ids = generate_text(
        model, sample_ids, max_new_tokens=40,
        context_length=model_config.context_length, temperature=0.8,
        top_k=40, eos_token_id=eos_token_id,
    )
    sample_output = tokenizer.decode(
        sample_output_ids[0].tolist(), skip_special_tokens=True
    )
    generation_samples.append({
        "prompt": sample_prompt,
        "generated_text": sample_output,
        "continuation": sample_output[len(sample_prompt):].lstrip()
            if sample_output.startswith(sample_prompt) else sample_output,
        "temperature": 0.8,
        "top_k": 40,
    })

evaluation_dir = CHECKPOINT_DIR.parent / "evaluation"
evaluation_dir.mkdir(parents=True, exist_ok=True)
tokenizer_sha256 = hashlib.sha256(TOKENIZER_PATH.read_bytes()).hexdigest()
evaluation_record = {
    "model_config": asdict(model_config),
    "training": {
        "max_steps": MAX_STEPS,
        "learning_rate": LEARNING_RATE,
        "seed": SEED,
    },
    "dataset": {
        "training_documents": len(train_documents),
        "validation_documents": len(validation_documents),
        "training_windows": len(train_dataset),
        "validation_windows": len(validation_dataset),
        "validation_target_tokens": len(validation_dataset) * CONTEXT_LENGTH,
    },
    "metrics": {
        "initial_validation_loss": initial_validation_loss,
        "best_validation_loss": final_validation_loss,
        "validation_perplexity": validation_perplexity,
    },
    "history": history,
    "checkpoint": str(checkpoint_path.relative_to(PROJECT_ROOT)),
    "tokenizer_sha256": tokenizer_sha256,
}
(evaluation_dir / "evaluation.json").write_text(
    json.dumps(evaluation_record, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
with (evaluation_dir / "generation_samples.jsonl").open(
    "w", encoding="utf-8", newline="\n"
) as stream:
    for sample in generation_samples:
        stream.write(json.dumps(sample, ensure_ascii=False) + "\n")
print("Saved evaluation report:", (evaluation_dir / "evaluation.json").relative_to(PROJECT_ROOT))
print("Saved generation samples:", len(generation_samples))

Prompt: ...عمر كان قالي تفاهمتو باش تتزوجو و دابا
Generated text: ...عمر كان قالي تفاهمتو باش تتزوجو و داباوش...قةط من من...ونرفكس تها على وك و ليش ل ع وم... م وشووتا...ز و ل...
Saved evaluation report: artifacts\evaluation\evaluation.json
Saved generation samples: 3


## What the evaluation tells us

This notebook has trained a small GPT from random weights and measured next-token loss and perplexity on the held-out split. The generated sample is a qualitative check. A stronger evaluation should compare checkpoints, inspect many prompts, and use a larger, carefully reviewed Darija test set. The tokenizer and checkpoint are local artifacts; the notebooks hold the code and learning sequence.